# 07 — Churn Prediction Model

The baseline is a class-weighted Random Forest. The evaluation split is chronological by snapshot date, not a random row split. This better reflects the intended future-scoring use case, although customers may appear in multiple historical snapshots.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / 'src').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from src.data_loader import load_olist_data
from src.analytics import prepare_order_level
from src.churn import (
    build_churn_snapshots, build_features_at_cutoff,
    train_and_score_churn
)

data = load_olist_data(ROOT / "data_raw")
order_level = prepare_order_level(data)
analysis_end = pd.Timestamp("2018-08-31 23:59:59")

training_snapshots = build_churn_snapshots(order_level, analysis_end, horizon_days=240)
scoring_features = build_features_at_cutoff(order_level, analysis_end, active_lookback_days=180)



## Fit, evaluate and score


In [ ]:
churn_predictions, churn_metrics, churn_importance = train_and_score_churn(
    training_snapshots,
    scoring_features,
    model_dir=str(ROOT / "models")
)

churn_metrics



In [ ]:
churn_importance.head(15)



In [ ]:
churn_predictions[
    ["customer_unique_id", "p_churn_240d", "risk_percentile", "risk_band"]
].head(20)



## Evaluation

Review average precision because the target may be imbalanced. ROC-AUC measures ranking across thresholds; Brier score measures probability error. Precision@10%, recall@10% and lift@10% describe how well the highest-risk decile concentrates churn outcomes.

Risk bands are relative ranking groups, not calibrated operational thresholds. A high churn probability does not prove that a customer can be saved; treatment uplift requires experiments.


## Actual temporal-holdout result

The churn model trained on 114,512 snapshots and tested on 54,458 snapshots (Dec 2017–Jan 2018). Test churn prevalence was 98.27%, ROC-AUC 0.612, and lift@10% 1.009. Average precision (0.9879) is close to the prevalence baseline (0.9827), so it must not be interpreted as strong performance. The model is not production-ready for CRM targeting.

In [ ]:
import json
with open(ROOT / "outputs" / "churn_model_metrics.json", encoding="utf-8") as f:
    churn_metrics = json.load(f)
display(pd.Series(churn_metrics))